# Air Pollutant Emissions Visualizations (Data1)

This notebook provides diverse visualizations for the emissions of air pollutants from all sectors dataset.

## Required Visualizations:
1. **Line Chart**: Global trends of key pollutants over time.
2. **Map**: Global choropleth map of Nitrogen oxides.
3. **Bar Chart**: Top 10 emitting countries for Sulfur dioxide.
4. **Table**: Summary statistics of pollutants by Entity.
5. **Scatter Plot**: Relationship between Nitrogen oxides and Sulfur dioxide.
6. **Stacked Area Chart**: Cumulative global pollutant emissions over time.

### 1. Line Chart: Global Trends of Key Pollutants
Visualizing how global emissions of SO2, NOx, and Methane have evolved since 1750.

In [ ]:
import os
import pandas as pd
import plotly.express as px

# Save plots under the workspace-level figures folder
base_dir = os.path.abspath(os.path.join(os.getcwd(), os.pardir))
output_dir = os.path.join(base_dir, 'figures', 'data1')
os.makedirs(output_dir, exist_ok=True)

def save_fig(fig, name):
    path = os.path.join(output_dir, name)
    fig.write_image(path + '.png')

# Load the dataset
data1_path = 'emissions-of-air-pollutants-from-all-sectors.csv'
df1 = pd.read_csv(data1_path)

# Filter out rows without ISO Codes (these are usually regions like "World")
df1_countries = df1.dropna(subset=['Code'])
df_world = df1[df1['Entity'] == 'World']

print("Data loaded successfully.")
df1.head()

Data loaded successfully.


,Entity,Code,Year,Ammonia,Black carbon,Carbon monoxide,Methane,Nitrogen oxides,Nitrous oxide,Non-methane volatile organic compounds,Organic carbon,Sulfur dioxide
0,Afghanistan,AFG,1750,7681.0464,1633.0308,142073.30,NaN,555.47860,NaN,13596.633,5456.8850,174.87167
1,Afghanistan,AFG,1751,7713.0280,1639.6868,142652.38,NaN,557.78180,NaN,13652.101,5479.1265,175.58444
2,Afghanistan,AFG,1752,7745.0103,1646.3417,143231.36,NaN,560.08510,NaN,13707.559,5501.3643,176.29706
3,Afghanistan,AFG,1753,7776.9917,1652.9952,143810.22,NaN,562.38824,NaN,13763.007,5523.5977,177.00955
4,Afghanistan,AFG,1754,7808.9720,1659.6476,144388.97,NaN,564.69130,NaN,13818.446,5545.8270,177.72192


In [2]:
pollutants = ['Sulfur dioxide', 'Nitrogen oxides', 'Methane']
fig_line = px.line(df_world, x='Year', y=pollutants, 
                  title='Global Trends of Key Pollutants (1750-Present)',
                  labels={'value': 'Emissions', 'variable': 'Pollutant'})
fig_line.show()
# save
save_fig(fig_line, 'global_trends')

### 2. Map: Global Choropleth of Nitrogen Oxides
Showing the distribution of Nitrogen oxides emissions for the most recent year.

In [3]:
latest_year = df1_countries['Year'].max()
df_latest = df1_countries[df1_countries['Year'] == latest_year]

fig_map = px.choropleth(df_latest, locations="Code",
                    color="Nitrogen oxides",
                    hover_name="Entity",
                    title=f"Global Nitrogen Oxides Emissions in {latest_year}",
                    color_continuous_scale=px.colors.sequential.Plasma)
fig_map.show()
# save
save_fig(fig_map, 'choropleth_nox')

### 3. Bar Chart: Top 10 Emitters for Sulfur Dioxide
Comparing the top emitters in the latest year.

In [4]:
top_10_so2 = df_latest.nlargest(10, 'Sulfur dioxide')

fig_bar = px.bar(top_10_so2, x='Entity', y='Sulfur dioxide',
                title=f'Top 10 Emitters of Sulfur Dioxide ({latest_year})')
fig_bar.show()
# save
save_fig(fig_bar, 'top10_so2')

### 4. Table: Summary Statistics by Entity
A detailed statistical view of the emissions data.

In [5]:
summary_table = df1_countries.groupby('Entity')[['Sulfur dioxide', 'Nitrogen oxides', 'Methane']].agg(['mean', 'max', 'min']).reset_index().head(20)
summary_table.columns = ['Entity', 'SO2_mean', 'SO2_max', 'SO2_min', 'NOx_mean', 'NOx_max', 'NOx_min', 'CH4_mean', 'CH4_max', 'CH4_min']

summary_plot = summary_table[['Entity', 'SO2_mean', 'NOx_mean', 'CH4_mean']].melt(
    id_vars='Entity',
    var_name='Pollutant',
    value_name='Average emissions'
 )
summary_plot['Pollutant'] = summary_plot['Pollutant'].map({
    'SO2_mean': 'Sulfur dioxide',
    'NOx_mean': 'Nitrogen oxides',
    'CH4_mean': 'Methane'
})

fig_table = px.bar(
    summary_plot,
    x='Entity',
    y='Average emissions',
    color='Pollutant',
    barmode='group',
    title='Average Pollutant Emissions by Entity (Top 20)',
    labels={'Entity': 'Entity', 'Average emissions': 'Average emissions'}
)

fig_table.update_layout(
    xaxis_tickangle=-45,
    margin=dict(l=20, r=20, t=60, b=100),
    height=560
)
fig_table.show()
# save
save_fig(fig_table, 'summary_by_entity')

### 5. Scatter Plot: Nitrogen Oxides vs Sulfur Dioxide
Examining the correlation between two major air pollutants across countries in the latest year.

In [6]:
fig_scatter = px.scatter(df_latest, x='Nitrogen oxides', y='Sulfur dioxide', 
                         hover_name='Entity', trendline="ols",
                         title='Correlation: Nitrogen Oxides vs Sulfur Dioxide')
fig_scatter.show()
# save
save_fig(fig_scatter, 'scatter_nox_so2')

### 6. Stacked Area Chart: Cumulative Global Emissions
Visualizing the share of different pollutants over time at a global level.

In [7]:
global_emissions = df_world[['Year', 'Sulfur dioxide', 'Nitrogen oxides', 'Methane']].copy()
global_emissions = global_emissions.groupby('Year', as_index=False).sum()

fig_table = px.area(
    global_emissions,
    x='Year',
    y=['Sulfur dioxide', 'Nitrogen oxides', 'Methane'],
    title='Cumulative Global Pollutant Emissions Over Time',
    labels={'value': 'Emissions', 'Year': 'Year', 'variable': 'Pollutant'}
)

fig_table.update_layout(
    hovermode='x unified',
    margin=dict(l=20, r=20, t=60, b=20),
    height=560
)
fig_table.show()
# save
save_fig(fig_table, 'cumulative_global_emissions')